# Test toàn bộ pipeline Agentic RAG (end-to-end)

Notebook này tách hệ thống thành **6 trạm dừng** để quan sát rõ từng giai đoạn biến đổi của một câu hỏi:

```
(1) Câu hỏi  →  (2) Embedding (vector 768d)  →  (3) Qdrant similarity + BM25 + RRF
             →  (4) Agent Analyzer (phân loại + chuẩn hoá + mở rộng truy vấn)
             →  (5) LangGraph Agent (routing → legal_rag / chit_chat / web / out_of_scope)
             →  (6) Câu trả lời + trích dẫn
```

**Tiền điều kiện**
- Qdrant đang chạy ở `localhost:6333` (collection `Traffic_Law_Hybrid` đã được index).
- File `.env` ở repo root chứa `API_KEY` (Gemini) và (tuỳ chọn) `TAVILY_API_KEY` cho nhánh web fallback.
- Venv `~/venv/LLM_Agentic` có sẵn các phụ thuộc trong `requirements.txt`.


## 0. Cấu hình hiển thị + nạp `.env`

In [ ]:
from IPython.display import display, HTML
display(HTML("<style>.jp-OutputArea-child { max-height: unset !important; } .jp-OutputArea-output { max-height: unset !important; }</style>"))

import os, sys
from pathlib import Path

BASE_DIR = Path().resolve().parent if Path().resolve().name in ("notebooks", "tests") else Path().resolve()
if str(BASE_DIR) not in sys.path:
    sys.path.insert(0, str(BASE_DIR))
if str(BASE_DIR / "source") not in sys.path:
    sys.path.insert(0, str(BASE_DIR / "source"))

for parent in [Path().resolve(), *Path().resolve().parents]:
    cand = parent / ".env"
    if cand.exists():
        for line in cand.read_text(encoding="utf-8").splitlines():
            line = line.strip()
            if not line or line.startswith("#") or "=" not in line:
                continue
            k, v = line.split("=", 1)
            os.environ[k.strip()] = v.strip().strip('"').strip("'")
        print(f"[OK] Nạp env từ {cand}")
        break

shared = os.environ.get("API_KEY")
if shared:
    os.environ.setdefault("GOOGLE_API_KEY", shared)
    os.environ.setdefault("GEMINI_API_KEY", shared)

print("GOOGLE_API_KEY:", "OK" if os.environ.get("GOOGLE_API_KEY") else "MISSING")
print("TAVILY_API_KEY:", "OK" if os.environ.get("TAVILY_API_KEY") else "MISSING (nhánh web fallback sẽ tắt)")

## 1. Câu hỏi đầu vào

Bạn có thể đổi `QUESTION` ở đây — toàn bộ notebook sẽ chạy theo câu hỏi này.

In [ ]:
QUESTION = "Vượt đèn đỏ khi điều khiển ô tô bị phạt bao nhiêu tiền và trừ bao nhiêu điểm GPLX?"
print("=" * 80)
print("Câu hỏi gốc:")
print(" ", QUESTION)
print("=" * 80)

## 2. Bước 1 — Câu hỏi → Vector Embedding

### 2.1. Vì sao là **768 chiều**?

Con số `768` không tự nhiên có — nó là **`hidden_size`** trong kiến trúc của model. `intfloat/multilingual-e5-base` bên trong là một biến thể của XLM-RoBERTa-base (12 layer transformer, 12 attention head, hidden_size = 768). Khi encode 1 câu, model chạy qua các lớp transformer rồi **mean-pool** các token vector lại thành **1 vector duy nhất có đúng `hidden_size` chiều**, tức 768.

Đối chiếu cả họ E5 để bạn thấy quy luật:

| Model | Backbone | Embedding dim |
|---|---|---|
| `multilingual-e5-small` | MiniLM-12 | **384** |
| `multilingual-e5-base`  | XLM-RoBERTa-base | **768** ← dự án đang dùng |
| `multilingual-e5-large` | XLM-RoBERTa-large | **1024** |

Đổi sang model khác → số chiều thay đổi → **collection Qdrant phải re-index** vì `vector size` được khoá khi tạo collection.

### 2.2. **L2 norm** là gì, vì sao có con số đó?

L2 norm (norm Euclid) = **độ dài** của vector trong không gian 768 chiều, tính theo Pythagoras tổng quát:

$$\lVert v \rVert_2 \;=\; \sqrt{v_1^2 + v_2^2 + \dots + v_{768}^2}$$

- Cell bên dưới gọi `encode(..., normalize_embeddings=False)` → **không** chia vector cho norm của nó → bạn thấy **norm thật** (E5 thường rơi vào khoảng **~12–18**, vì đầu ra mean-pool của 768 toạ độ phân bố gần $\mathcal{N}(0,\sigma^2)$ cho ra norm cỡ $\sqrt{768}\cdot\sigma$).
- Nếu đặt `normalize_embeddings=True`, SBERT chia mọi thành phần cho norm trước khi trả ra → norm bằng **đúng 1.0**, vector nằm trên mặt cầu đơn vị.
- **Vì sao retriever production lại normalize?** Cosine = $\dfrac{\langle a,b\rangle}{\lVert a\rVert\,\lVert b\rVert}$, magnitude bị triệt tiêu nên về toán học bằng nhau. Nhưng nếu cả `a` và `b` đã normalize, Qdrant chỉ cần làm `dot(a,b)` → tiết kiệm 1 phép chia mỗi điểm × hàng nghìn điểm = nhanh hơn rõ rệt. **Tối ưu hiệu năng, không đổi thứ tự ranking.**

### 2.3. Vì sao E5 cần prefix `"query: "`?

E5 được huấn luyện **asymmetric**: cặp dữ liệu lúc train luôn có dạng `"query: <câu hỏi>"` ↔ `"passage: <đoạn văn>"`. Model học cách kéo 2 vector này gần nhau khi liên quan. Nếu lúc inference quên prefix → vector câu hỏi rơi vào vùng phân phối khác vùng đã index → **chất lượng top-k giảm rõ rệt** (paper E5 báo nDCG@10 giảm 5–10%).

Cell bên dưới in **shape, dtype, L2 norm và 16 chiều đầu** để bạn đối chiếu 3 ý trên.

In [ ]:
import numpy as np
from sentence_transformers import SentenceTransformer

EMBED_MODEL = "intfloat/multilingual-e5-base"
print(f"Đang nạp model embedding: {EMBED_MODEL} ...")
embedder = SentenceTransformer(EMBED_MODEL)

query_text = f"query: {QUESTION}"   # prefix bắt buộc cho e5
vec = embedder.encode(query_text, normalize_embeddings=False)
vec = np.asarray(vec, dtype=np.float32)

print("\nshape :", vec.shape, "  ← 768 = hidden_size của XLM-RoBERTa-base")
print("dtype :", vec.dtype)
print("L2 norm:", float(np.linalg.norm(vec)), "  ← sqrt(sum(v_i^2)) trên cả 768 chiều")

# Đối chiếu: nếu normalize thì norm phải = 1.0
vec_n = embedder.encode(query_text, normalize_embeddings=True)
print("L2 norm (normalize=True):", float(np.linalg.norm(vec_n)), "  ← gần 1.0 đúng như kỳ vọng")

print("\n16 chiều đầu của vector (chưa normalize):")
print(np.round(vec[:16], 4).tolist())

## 3. Bước 2 — So sánh vector với Qdrant (similarity thuần)

### 3.1. Số `points_count` (vd. **3575**) ở đâu ra?

Đó là **số chunk đã được index** trong collection — mỗi point trong Qdrant = 1 chunk văn bản pháp luật + vector 768d + payload (metadata: `doc_id`, `dieu`, `khoan`, `diem`, `topic`, `status`...).

Pipeline tạo ra con số này:

1. **Crawl + clean**: PDF/Word của các Luật, Nghị định, Thông tư → file Markdown sạch trong `traffic_rag/Data/cleaned/`.
2. **Chunking**: `source/ingestion/fixed_size_chunker.py` (hoặc semantic chunker) tách mỗi văn bản thành các đoạn cỡ ~512 token, **gắn kèm tiêu đề Điều/Khoản** vào đầu mỗi đoạn để giữ ngữ cảnh — file kết quả: `Data/all_chunks.jsonl`. Tổng số dòng JSONL chính là số points sẽ index.
3. **Embed + upsert**: `source/indexing/indexer.py` đọc JSONL, gọi `model.encode(passage_text)` cho từng dòng, đẩy lên Qdrant theo **id = chỉ số dòng** (insertion order). Đây là điểm then chốt: BM25 in-memory cũng dùng cùng index → **cùng `id` ở cả 2 retriever** → fuse được mà không cần lớp ánh xạ.

Số 3575 (hoặc bất cứ số nào bạn thấy ở cell dưới) = số dòng `all_chunks.jsonl` tại lần index gần nhất. Re-ingest nhiều văn bản hơn → số tăng; đổi chunker (vd. fixed-256 thay vì fixed-512) → cũng đổi.

### 3.2. **Top-5 sinh ra như thế nào?**

`client.search(query_vector=v, limit=5)` thực hiện **Approximate Nearest Neighbor** trong Qdrant:

1. Qdrant index dùng **HNSW** (Hierarchical Navigable Small World) — đồ thị nhiều tầng nối các vector "láng giềng". Search bắt đầu từ một entry point ở tầng cao nhất, đi xuống dần và **greedy** chọn neighbor có cosine cao nhất với `v`.
2. Mỗi điểm `p` được chấm điểm `cosine(v, p) = dot(v, p) / (|v|·|p|)`. Vì collection được tạo với `Distance.COSINE`, Qdrant trả `score ∈ [-1, 1]` (càng gần 1 càng tương đồng). Trong miền văn bản pháp luật tiếng Việt, score điển hình rơi vào **0.75 – 0.95** cho top-1.
3. Tập kết quả được giữ trong 1 priority queue kích thước `ef` (mặc định ~64); cuối cùng trả về `limit=5` điểm có score cao nhất.
4. **Vì sao là approximate, không exact?** Brute force cosine với 3.5k điểm × 768 chiều cũng nhanh, nhưng HNSW cho phép scale lên hàng triệu điểm vẫn < 10ms. Tradeoff: ~99% recall@10 trong cấu hình mặc định — hoàn toàn chấp nhận được cho RAG.

Đây mới chỉ là **tầng dense**. Có những câu hỏi dùng khẩu ngữ ("vượt đèn đỏ") mà luật chính thức ghi cụm khác ("không chấp hành hiệu lệnh đèn tín hiệu") → cosine không đủ, cần thêm BM25 ở bước 4.

In [ ]:
from qdrant_client import QdrantClient

QDRANT_HOST = "localhost"
QDRANT_PORT = 6333
COLLECTION  = "Traffic_Law_Hybrid"

client = QdrantClient(host=QDRANT_HOST, port=QDRANT_PORT)
info = client.get_collection(COLLECTION)
print(f"Collection: {COLLECTION}")
print(f"  points_count: {info.points_count}   ← số chunk đã index = số dòng all_chunks.jsonl")
print(f"  vectors cfg : {info.config.params.vectors}   ← phải có size=768, distance=Cosine")

TOP_K = 5
hits = client.search(
    collection_name=COLLECTION,
    query_vector=vec.tolist(),
    limit=TOP_K,
    with_payload=True,
)

print(f"\nTop-{TOP_K} chunks gần nhất theo cosine (chỉ dense, chưa fuse với BM25):\n")
for i, h in enumerate(hits, 1):
    md = h.payload or {}
    print(f"[{i}] cosine={h.score:.4f}  id={h.id}  doc={md.get('doc_id','?')}  Điều {md.get('dieu','?')}")
    text = (md.get("content") or md.get("text") or "")[:160].replace("\n", " ")
    print(f"    {text}...\n")

## 4. Bước 3 — Hybrid Retriever: Dense + BM25 + RRF

### 4.1. Vì sao cần **2 retriever**?

| Yếu tố | Dense (vector) | BM25 (lexical) |
|---|---|---|
| Hiểu **nghĩa** đồng nghĩa, paraphrase | ✅ tốt | ❌ kém |
| Khớp **chính xác từ khoá**: số văn bản, tên Điều, mã GPLX | ⚠️ dễ trượt | ✅ rất tốt |
| Câu hỏi khẩu ngữ vs văn bản chính thống | ✅ kéo gần | ❌ không match nếu khác từ |
| Hiệu năng | tốt với HNSW | rất tốt (in-memory, 3.5k docs) |

→ Kết hợp 2 cái mới cover được cả 2 phía. Đây là pattern chuẩn của các hệ RAG production (ColBERT, Vespa, Elasticsearch hybrid, v.v.).

### 4.2. **Dense — chi tiết**

Đã giải thích ở section 3: encode bằng E5, search HNSW trên Qdrant, trả `candidates_per_retriever=30` ứng viên (không phải `top_k=5` cuối) để có nhiều "đạn" cho fusion ở bước sau.

### 4.3. **BM25 — chi tiết**

BM25 (Best Match 25) là biến thể của TF-IDF, công thức cho 1 truy vấn $Q = \{q_1, q_2, \dots\}$ và document $D$:

$$\mathrm{BM25}(D, Q) \;=\; \sum_{q \in Q} \mathrm{IDF}(q) \cdot \frac{f(q, D)\,(k_1 + 1)}{f(q, D) + k_1\!\left(1 - b + b\,\dfrac{|D|}{\mathrm{avgdl}}\right)}$$

trong đó $f(q,D)$ là số lần $q$ xuất hiện trong $D$, $|D|$ là độ dài document, `avgdl` là độ dài trung bình corpus. `k1≈1.5`, `b≈0.75` (mặc định của `rank_bm25`).

Trong dự án, `_build_bm25_corpus` ([retriever.py:242](../source/rag_core/retriever.py#L242)) làm 3 việc:

1. **Đọc `Data/all_chunks.jsonl`** — đúng file mà indexer đã đẩy lên Qdrant → cùng số dòng, cùng thứ tự.
2. **Tokenize tiếng Việt nhẹ**: lower-case, bỏ dấu câu, lọc stopword (`là, và, của, cho, ...`), bỏ token ≤ 1 ký tự. *Không* dùng underthesea/pyvi (tránh phụ thuộc nặng); thuật ngữ pháp luật tiếng Việt phần lớn là từ ghép có dấu nên tokenize bằng khoảng trắng vẫn đủ tốt.
3. **`BM25Okapi(tokenized)`** — dựng index in-memory. Mỗi truy vấn gọi `bm25.get_scores(q_tokens)` trả về vector điểm có độ dài bằng số document, ta `argsort` để lấy top-30.

### 4.4. **RRF — Reciprocal Rank Fusion** (`rrf_k = 60`)

Vấn đề khi fuse: cosine ∈ [-1, 1], BM25 ∈ [0, +∞). **Cộng score thẳng** sẽ bị BM25 át hoàn toàn dense (hoặc ngược lại sau khi normalize vụng). RRF giải bằng cách **bỏ score đi, chỉ dùng rank**:

$$\mathrm{RRF}(d) \;=\; \sum_{i \in \text{retrievers}} \frac{1}{k + \mathrm{rank}_i(d)}$$

Với mỗi document $d$, lấy thứ hạng của nó trong từng retriever (1 là cao nhất), cộng nghịch đảo có shift bằng `k`.

**Vì sao `k = 60`?** Đây là giá trị **mặc định kinh điển** từ paper gốc Cormack–Clarke–Buettcher (SIGIR 2009: *Reciprocal Rank Fusion outperforms Condorcet and individual Rank Learning Methods*). Trực giác:

| `k` | Hiệu ứng |
|---|---|
| **k nhỏ (vd. 10)** | rank cao đè rất mạnh — top-1 thắng tuyệt đối, ít cơ hội cho doc xếp hạng vừa nhưng "đồng thuận" giữa 2 retriever nhảy lên. |
| **k = 60** ← chuẩn | 1/(60+1)=0.0164 vs 1/(60+10)=0.0143 — chênh lệch các rank cao là **vừa đủ** để doc rank trung bình ở cả 2 retriever vẫn vượt được doc top-1 chỉ ở 1 retriever. Đây chính là điểm mạnh của hybrid. |
| **k lớn (vd. 200)** | rank gần như mất ý nghĩa — mọi doc gần ngang điểm. |

Ví dụ minh hoạ với `k=60`:

- Doc A: rank 1 ở Dense, rank 1 ở BM25 → score = `1/61 + 1/61 = 0.0328`
- Doc B: rank 1 ở Dense, không có ở BM25 → `1/61 + 0 = 0.0164`
- Doc C: rank 5 ở Dense, rank 3 ở BM25 → `1/65 + 1/63 = 0.0313`

→ A > C > B. Doc C "đồng thuận trung bình" thắng được Doc B "chỉ tốt ở 1 phía". Đó là lý do RRF đặc biệt ổn định cho hybrid.

Cài đặt thực tế ở [retriever.py:369](../source/rag_core/retriever.py#L369): `_rrf_fuse(dense_hits, bm25_hits, top_k)` — gom điểm bằng `dict[id] += 1/(k+rank)` rồi sort giảm dần.

### 4.5. Sibling enrichment (bonus)

Sau RRF, `_attach_siblings` còn kéo thêm các Khoản kề và các đoạn "completion clause" (bảng trừ điểm GPLX, hình thức xử phạt bổ sung) cho cùng Điều — vì câu trả lời mức phạt trong NĐ 168 **luôn nằm rải ở 2-3 chunk** chứ không gói trọn trong 1 chunk. Đó là lý do cell dưới có thể trả về > top_k dòng.

In [ ]:
from rag_core import TrafficHybridRetriever

retriever = TrafficHybridRetriever()
print(f"rrf_k                    = {retriever.rrf_k}")
print(f"candidates_per_retriever = {retriever.candidates_per_retriever}  (lấy 30 ứng viên mỗi retriever trước khi fuse)")
print(f"BM25 docs                = {len(retriever._payloads)}              (= points_count ở trên)")
print()

chunks = retriever.get_relevant_chunks(QUESTION, top_k=5)
print(f"Hybrid retriever trả {len(chunks)} chunks (đã dedup + sibling enrichment):\n")
for i, c in enumerate(chunks[:8], 1):
    md = c.metadata
    sib = "  [sibling]" if md.get("is_sibling") else ""
    print(f"[{i}] RRF={c.score:.4f}  doc={md.get('doc_id','?')}  Điều {md.get('dieu','?')}{sib}")
    print(f"    {c.content[:160].replace(chr(10),' ')}...\n")

## 5. Bước 4 — Agent Analyzer (phân loại + chuẩn hoá + mở rộng query)

Trước khi đi vào graph, `analyzer_node` gọi LLM 1 lần duy nhất để:
1. **Phân loại** vào `legal_rag` / `chit_chat` / `web_legal_search` / `out_of_scope`.
2. **Chuẩn hoá** (`standalone_query`): viết lại câu độc lập, giải tham chiếu ("xe đó" → "xe ô tô con"...).
3. **Mở rộng** (`expanded_query`): thêm thuật ngữ pháp lý chính thức để retriever tìm trúng — đây mới là chuỗi đem đi search ở bước sau.

In [ ]:
from langchain_google_genai import ChatGoogleGenerativeAI
from agent.nodes import make_analyzer_node

GEN_MODEL = "gemini-3.1-flash-lite-preview"

llm = ChatGoogleGenerativeAI(
    model=GEN_MODEL,
    temperature=0.1,
    google_api_key=os.environ["GOOGLE_API_KEY"],
    request_timeout=30,
)

analyzer = make_analyzer_node(llm)
analyzer_state = {"query": QUESTION, "chat_history": []}
out = analyzer(analyzer_state)

print("--- KẾT QUẢ ANALYZER ---")
print("category         :", out.get("category"))
print("raw_query        :", out.get("raw_query"))
print("standalone_query :", out.get("query"))
print("expanded_query   :", out.get("expanded_query"))

### 5.1 — So sánh retrieval với `expanded_query`

Đem `expanded_query` từ bước 5 đi search lại để xem mức độ "trúng đích" cải thiện ra sao so với câu hỏi thô ở bước 4.

In [ ]:
expanded = out.get("expanded_query") or QUESTION
chunks_exp = retriever.get_relevant_chunks(expanded, top_k=5)

print(f"Top-5 với expanded_query (RRF):\n")
for i, c in enumerate(chunks_exp, 1):
    md = c.metadata
    print(f"[{i}] RRF={c.score:.4f}  doc={md.get('doc_id','?')}  Điều {md.get('dieu','?')}")
    print(f"    {c.content[:160].replace(chr(10),' ')}...\n")

## 6. Bước 5 — Build Generator + LangGraph Agent

Lắp đầy đủ graph như API production (`api/main.py`), nhưng **tắt HITL interrupt** (`enable_hitl_interrupt=False`) để chạy end-to-end trong notebook không cần phê duyệt thủ công.

Nếu không có `TAVILY_API_KEY` ta truyền `tavily_tool=None` — graph vẫn chạy được nhánh `legal_rag` / `chit_chat` / `out_of_scope`, chỉ là khi router đưa vào `web_search` thì sẽ raise.

In [ ]:
from rag_core import LegalAnswerGenerator
from agent import build_graph, TavilySearchTool

generator = LegalAnswerGenerator(provider="google", model=GEN_MODEL)

tavily = None
if os.environ.get("TAVILY_API_KEY"):
    try:
        tavily = TavilySearchTool()
        print("[OK] Tavily web-search tool đã sẵn sàng.")
    except Exception as e:
        print(f"[WARN] Không khởi tạo được Tavily: {e}")
else:
    print("[INFO] TAVILY_API_KEY không có — nhánh web_search sẽ không khả dụng.")

graph = build_graph(
    retriever=retriever,
    generator=generator,
    llm=llm,
    tavily_tool=tavily,
    checkpoint_db=str(BASE_DIR / "checkpoints" / "notebook_test.db"),
    enable_hitl_interrupt=False,   # chạy thẳng, không pause để approve
)
print("[OK] Graph compiled.")

## 7. Bước 6 — Chạy graph end-to-end

`graph.invoke(...)` sẽ chạy chuỗi node: `analyzer → (route) → legal_rag → END` và trả về toàn bộ `AgentState`. Mỗi `thread_id` ứng với một phiên hội thoại được lưu trong SQLite checkpointer (giống production).

In [ ]:
import uuid

config = {"configurable": {"thread_id": f"nb-{uuid.uuid4().hex[:8]}"}}

final_state = graph.invoke(
    {"query": QUESTION, "chat_history": []},
    config=config,
)

print("=" * 80)
print("thread_id   :", config["configurable"]["thread_id"])
print("category    :", final_state.get("category"))
print("refused     :", final_state.get("refused"))
print("#chunks     :", len(final_state.get("chunks") or []))
print("model       :", final_state.get("model_info"))
print("=" * 80)
print("\n========= CÂU TRẢ LỜI =========\n")
print(final_state.get("answer") or "(rỗng)")
print("\n========= TRÍCH DẪN =========\n")
for s in (final_state.get("sources") or []):
    parts = [s.get("doc_id", "?"), f"Điều {s.get('dieu','?')}"]
    if s.get("khoan") is not None: parts.append(f"Khoản {s['khoan']}")
    if s.get("diem")  is not None: parts.append(f"Điểm {s['diem']}")
    print("  -", " | ".join(parts), "—", s.get("ten_van_ban", ""))

## 8. Trace từng node (debug)

Dùng `graph.stream(...)` thay cho `invoke` để **xem state tăng dần qua từng node**. Hữu ích khi muốn biết node nào đã ghi field nào, hoặc query đi qua đường nào trong 4 nhánh.

In [ ]:
config2 = {"configurable": {"thread_id": f"nb-trace-{uuid.uuid4().hex[:8]}"}}

print(f"[TRACE] thread_id={config2['configurable']['thread_id']}\n")
for step in graph.stream(
    {"query": QUESTION, "chat_history": []},
    config=config2,
    stream_mode="updates",
):
    for node_name, delta in step.items():
        keys = list(delta.keys()) if isinstance(delta, dict) else type(delta).__name__
        print(f"→ NODE: {node_name}")
        print(f"   updates keys: {keys}")
        if isinstance(delta, dict):
            for k in ("category", "expanded_query", "refused", "answer"):
                if k in delta:
                    val = delta[k]
                    if isinstance(val, str) and len(val) > 200:
                        val = val[:200] + "…"
                    print(f"   {k}: {val}")
        print()

## 9. Test các nhánh routing khác

Cùng một graph, khác câu hỏi → router đưa vào các nhánh khác nhau:
- `chit_chat` (chào hỏi)
- `out_of_scope` (không liên quan giao thông)
- `legal_rag` (mặc định cho luật giao thông VN)

*Ghi chú:* nhánh `web_legal_search` cần `TAVILY_API_KEY` thật mới chạy được; nếu không có, để `enable_hitl_interrupt=False` thì cũng không pause, nhưng tool gọi sẽ thất bại — bỏ qua case đó nếu chưa cấu hình.

In [ ]:
import time

ROUTING_CASES = [
    ("chit_chat (kỳ vọng)",   "Xin chào, bạn là ai?"),
    ("out_of_scope (kỳ vọng)", "Cho tôi công thức nấu phở bò"),
    ("legal_rag (kỳ vọng)",   "Nồng độ cồn vượt 0,4 mg/lít khí thở khi đi xe máy phạt bao nhiêu?"),
]

for label, q in ROUTING_CASES:
    print("=" * 80)
    print(f"[CASE] {label}")
    print(f"Q: {q}")
    cfg = {"configurable": {"thread_id": f"nb-route-{uuid.uuid4().hex[:8]}"}}
    try:
        st = graph.invoke({"query": q, "chat_history": []}, config=cfg)
        print(f"category : {st.get('category')}")
        ans = st.get("answer") or ""
        print(f"answer   : {ans[:300]}{'…' if len(ans) > 300 else ''}")
    except Exception as e:
        print(f"[ERROR] {type(e).__name__}: {e}")
    print()
    time.sleep(4)  # tránh đụng rate limit free tier 15 RPM

## 10. Tóm tắt — bạn vừa quan sát điều gì?

| Bước | Đầu vào | Đầu ra | File nguồn |
|---|---|---|---|
| 1 | Câu hỏi text | Vector 768d (E5) | `source/rag_core/retriever.py` |
| 2 | Vector | Top-k cosine từ Qdrant | (Qdrant client) |
| 3 | Câu hỏi | Top-k chunks RRF (Dense+BM25) | `source/rag_core/retriever.py` |
| 4 | Câu hỏi + history | category, standalone, expanded | `source/agent/nodes.py:make_analyzer_node` |
| 5 | AgentState | Câu trả lời + sources | `source/agent/graph.py:build_graph` |
| 6 | Stream updates | Trace từng node | `graph.stream(...)` |

**Các con số quan trọng đã giải thích:**

- **768** = `hidden_size` của XLM-RoBERTa-base bên trong `multilingual-e5-base` (section 2.1).
- **L2 norm ~ 12–18** = chiều dài Euclid của vector chưa normalize; nếu normalize → đúng 1.0 (section 2.2).
- **points_count (~3575)** = số chunk đã index trong Qdrant, bằng số dòng `Data/all_chunks.jsonl` (section 3.1).
- **top-5** = HNSW approximate nearest neighbor trên cosine, lấy 5 điểm có score cao nhất (section 3.2).
- **rrf_k = 60** = hằng số chuẩn từ paper RRF gốc (Cormack 2009); cân bằng giữa "top rank thắng đậm" và "đồng thuận giữa các retriever" (section 4.4).
- **candidates_per_retriever = 30** = mỗi retriever đưa 30 ứng viên vào fusion để có đủ overlap.
